# 第 2 章 §2.4 —— Google ADK（Auto-Flow 隐式路由）逐段拆解

> 这是书里 `Chapter_02_Routing_(Google_ADK)` 的**注释版 + 本机适配版**。
> 原代码在 ADK 2.13.0 上**跑不起来**，有三处必须改（见下表）。逻辑未变。

## 它和 §2.3 的根本区别

| | §2.3 LangChain | §2.4 Google ADK |
|---|---|---|
| 路由逻辑在哪 | **你的代码里**（`RunnableBranch` 的 lambda） | **模型自己决定** |
| 代码里有 if/else | 有 | **一个都没有** |
| 模型做什么 | 只输出一个标签字符串 | 读 description，决定**转交给谁** |
| 触发条件 | 你写分支 | `sub_agents=[...]` 存在即自动开启 |
| 模型调用数 | 1 次 | **≥2 次**（协调者 1 次 + 子 agent 1 次）|

一句话：**§2.3 你当调度员；§2.4 你写招聘启事，让模型自己选人。**

## 本机适配清单（三处，缺一不可）

| # | 问题 | 原代码 | 改为 |
|---|---|---|---|
| 1 | 模型已下线 | `gemini-2.0-flash` | **`gemini-3.1-flash-lite`**（实测可用，免费层额度较高）|
| 2 | `create_session` 是协程 | `session_service.create_session(...)` | **`create_session_sync(...)`** |
| 3 | 免费层限流 5 RPM | 连续跑 4 个用例 | 用例间 **sleep** + 429 自动重试 |

> 第 2 条是原文**直接报错**的原因：不 await 就创建不了 session，
> 之后 `runner.run` 报 `Session not found`。

## 可用模型实测（本机，2026-10-29）

| 模型 | 结果 |
|---|---|
| `gemini-3.1-flash-lite` | ✅ 可用（本 notebook 默认） |
| `gemini-3.5-flash` | ✅ 可用 |
| `gemini-3-flash-preview` | ✅ 可用 |
| `gemini-2.0-flash`（书里用的） | ❌ 已下线 |
| `gemini-2.5-flash` / `-lite` | ❌ 对新用户关闭 |
| `gemini-3.8-flash` / `3.7-flash` | ⚠️ 503 高负载（可稍后重试） |

## 读法

1. 第一节代码已逐段注释
2. 第二节讲 Auto-Flow **内部到底发生了什么**
3. 第三节是 4 个教科书里没有的实测发现

In [ ]:
# =============================================================================
# 第 2 章 §2.4 完整代码（逐行注释 + 本机适配）
# =============================================================================
#
# 先看清一件事：下面这整段代码里，**没有任何路由判断**。
# 没有 if 分支，没有标签比较，没有分类器调用。
# 你和模型之间的"契约"只有三样东西：
#   ① 协调者的 instruction（自然语言职责说明）
#   ② 每个子 agent 的 description（自然语言能力说明）
#   ③ sub_agents=[...]（把候选集告诉框架）
# 剩下的交给 ADK 的 Auto-Flow。

import os
import time
import uuid

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types

# 【适配 1】书里用 gemini-2.0-flash，该模型已下线；从环境变量读，便于换模型
MODEL = os.getenv("GOOGLE_MODEL", "gemini-3.1-flash-lite")
print("使用模型:", MODEL)


# ─────────────────────────────────────────────────────────────────────────────
# ① 三个工具实现 —— 注意：**docstring 就是工具描述**
# ─────────────────────────────────────────────────────────────────────────────
# FunctionTool 会把函数的 docstring 拿去做成工具的 description，
# 而 description 正是模型判断"该不该调用这个工具"的依据。
# 也就是说：**docstring 也是路由逻辑**，写不清楚，模型就不会调。
# 这和 §2.3 里 prompt 的作用一模一样，只是位置换了。
#
# 另外：类型注解（request: str -> str）会被转成 JSON Schema，
# 模型靠它决定传什么参数。少写注解，工具调用就容易出错。

def booking_handler(request: str) -> str:
    """Handles booking requests for flights and hotels.

    Args:
        request: The user's request for a booking.

    Returns:
        A confirmation message that the booking was handled.
    """
    print("  >>> Booking Handler Called")
    return f"Booking action for '{request}' has been simulated."


def info_handler(request: str) -> str:
    """Handles general information requests.

    Args:
        request: The user's question.

    Returns:
        A message indicating the information request was handled.
    """
    print("  >>> Info Handler Called")
    return f"Information request for '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    print("  >>> Unclear Handler Called")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

# ⚠️ 注意 unclear_handler：它在整段代码里**从未被注册成工具、也没被挂到任何 agent 上**。
# 书里第 11 页自己承认了："although the current coordinator logic doesn't
# explicitly use it for delegation failure"。
# 这就是隐式路由的代价 —— §2.3 里它至少还是 RunnableBranch 的默认分支，
# 这里连"兜底"这个概念都消失了。模型转交失败会怎样，代码没有定义。


# ─────────────────────────────────────────────────────────────────────────────
# ② 把函数包装成 Tool —— 这一步让模型"看得见"这些能力
# ─────────────────────────────────────────────────────────────────────────────
booking_tool = FunctionTool(booking_handler)
info_tool = FunctionTool(info_handler)


# ─────────────────────────────────────────────────────────────────────────────
# ③ 定义子 agent —— 每个都是"一个有 LLM、有工具、能独立决策的专家"
# ─────────────────────────────────────────────────────────────────────────────
# 关键字段是 description：Auto-Flow 就是靠它来挑人的。
# 换句话说，**description 写错，路由就错**，而且和 §2.3 改 prompt 一样：
# 没有 diff、没有测试会失败。
booking_agent = Agent(
    name="Booker",
    model=MODEL,
    description="A specialized agent that handles all flight and hotel booking requests by calling the booking tool.",
    tools=[booking_tool],
)

info_agent = Agent(
    name="Info",
    model=MODEL,
    description="A specialized agent that provides general information and answers user questions by calling the info tool.",
    tools=[info_tool],
)


# ─────────────────────────────────────────────────────────────────────────────
# ④ 协调者 —— 全篇的机关就在最后一行
# ─────────────────────────────────────────────────────────────────────────────
coordinator = Agent(
    name="Coordinator",
    model=MODEL,
    instruction=(
        "You are the main coordinator. Your only task is to analyze incoming user requests "
        "and delegate them to the appropriate specialist agent. Do not try to answer the user directly.\n"
        "- For any requests related to booking flights or hotels, delegate to the 'Booker' agent.\n"
        "- For all other general information questions, delegate to the 'Info' agent."
    ),
    description="A coordinator that routes user requests to the correct specialist agent.",
    # ★ 这一行是开关：只要存在 sub_agents，ADK 的 Auto-Flow 就自动启用，
    #   模型获得"把对话转交给某个子 agent"的能力。
    #   删掉它，coordinator 就退化成普通 agent，只会自己硬答。
    sub_agents=[booking_agent, info_agent],
)


# ─────────────────────────────────────────────────────────────────────────────
# ⑤ 执行器：建 session → 跑 → 从事件流里捞最终回复
# ─────────────────────────────────────────────────────────────────────────────
def extract_text(content) -> str:
    """从 event.content 里取文本。

    书里原版对这两种情况都做了处理 —— 不是多余的：
    本机实测 `Content` 对象**没有 `.text` 属性**，
    所以必须走 parts 分支。少写这段就会 AttributeError。
    """
    if getattr(content, "text", None):
        return content.text
    parts = getattr(content, "parts", None)
    if parts:
        return "".join(p.text for p in parts if getattr(p, "text", None))
    return ""


import re


def _retry_delay(msg: str, default: int = 30) -> int:
    """从 429 错误体里抠出服务端建议的等待秒数（形如 retryDelay: '18s'）。"""
    m = re.search(r"retryDelay['\"]?\s*:\s*['\"]?(\d+(?:\.\d+)?)s", msg)
    if m:
        return min(int(float(m.group(1))) + 5, 120)   # 留 5s 余量
    return default


def run_coordinator(runner: InMemoryRunner, request: str, *,
                    max_retry: int = 2, verbose: bool = False):
    """跑一个请求，从事件流里取出最终回复。

    ⚠️ 相对上一版修了三个真问题（它们会集中表现为「用例 A 总失败」）：
      1. 重试时**复用同一个 session** —— 上一次失败已经把用户消息写进去了，
         重试等于把同一句话再发一遍，还可能撞上"半截回合"。
         现在每次尝试都用全新 session。
      2. 固定等 45s —— 免费层是 60s 窗口，经常不够。
         现在优先读服务端返回的 retryDelay。
      3. 拿不到 final response 时静默返回 ""，看起来像"没反应"。
         现在会提示收到了多少事件，并支持 verbose 打出事件流。
    """
    print(f"\n--- Running Coordinator with request: '{request}' ---")
    user_id = "user_123"

    for attempt in range(1, max_retry + 2):
        # ★ 每次尝试一个全新 session（见上面第 1 点）
        session_id = str(uuid.uuid4())
        runner.session_service.create_session_sync(
            app_name=runner.app_name, user_id=user_id, session_id=session_id
        )
        try:
            final = None
            seen = 0
            for event in runner.run(
                user_id=user_id,
                session_id=session_id,
                new_message=types.Content(role="user", parts=[types.Part(text=request)]),
            ):
                seen += 1
                if verbose:
                    author = getattr(event, "author", "?")
                    print(f"   [event {seen}] author={author} final={event.is_final_response()}")
                if event.is_final_response() and event.content:
                    final = extract_text(event.content)
                    break

            if final is None:
                print(f"   ⚠️ 收到 {seen} 个事件，但没有 final response")
                print("      用 verbose=True 再跑一次可以看清事件流")
                return ""

            print(f"Coordinator Final Response: {final}")
            return final

        except Exception as e:
            msg = str(e)
            head = (msg.strip().splitlines() or [type(e).__name__])[0][:160]

            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                if attempt <= max_retry:
                    wait = _retry_delay(msg)
                    print(f"   ⏳ 第 {attempt} 次撞限流(429)，按服务端建议等 {wait}s 后重试（新 session）")
                    time.sleep(wait)
                    continue
                print(f"   ❌ 限流未恢复：{head}")
                return f"[RATE LIMITED] {head}"

            print(f"   ❌ 出错({type(e).__name__}): {head}")
            return f"[ERROR] {head}"

    return ""


# ─────────────────────────────────────────────────────────────────────────────
# ⑥ 演示
# ─────────────────────────────────────────────────────────────────────────────
runner = InMemoryRunner(coordinator)

# 书里原本跑 4 个用例。免费层 5 RPM 撑不住，这里默认只跑 2 个，
# 并在中间 sleep 25 秒，让配额窗口滑过去。
# 调试单点时把 VERBOSE 改成 True，可以看到每个事件是谁产生的。
# 想跑第 3、4 个，取消注释即可 —— 但请确保间隔足够。
DEMO = [
    ("A 预订", "Book me a hotel in Paris."),
    ("B 信息", "What is the highest mountain in the world?"),
    # ("C 信息", "Tell me a random fact."),
    # ("D 预订", "Find flights to Tokyo next month."),
]

VERBOSE = False        # ← 想看事件流就把这里改成 True

for i, (label, req) in enumerate(DEMO):
    if i > 0:
        print("\n(等待 25s 避开免费层限流...)")
        time.sleep(25)
    print(f"\n=== 用例 {label} ===")
    print("结果:", run_coordinator(runner, req, verbose=VERBOSE))

使用模型: gemini-3.1-flash-lite

=== 用例 A 预订 ===

--- Running Coordinator with request: 'Book me a hotel in Paris.' ---


  >>> Booking Handler Called


Coordinator Final Response: OK. I have processed your request to book a hotel in Paris.
结果: OK. I have processed your request to book a hotel in Paris.

(等待 20s 避开免费层限流...)



=== 用例 B 信息 ===

--- Running Coordinator with request: 'What is the highest mountain in the world?' ---


  >>> Info Handler Called


Coordinator Final Response: The highest mountain in the world is Mount Everest, which stands at 8,848.86 meters (29,031.7 feet) above sea level.
结果: The highest mountain in the world is Mount Everest, which stands at 8,848.86 meters (29,031.7 feet) above sea level.


## Auto-Flow 内部到底发生了什么

以 `"Book me a hotel in Paris."` 为例，一次调用产生的**事件序列**：

| # | 事件 | 谁产生的 | 花 token |
|---|---|---|---|
| 1 | 用户消息进入 Coordinator | — | ❌ |
| 2 | **Coordinator 的 LLM 决策**：我应该转交给 Booker | Coordinator | ✅ **调用 1** |
| 3 | 转交事件（transfer to Booker） | 框架 | ❌ |
| 4 | **Booker 的 LLM 决策**：我应该调用 booking_tool | Booker | ✅ **调用 2** |
| 5 | 工具调用 `booking_handler("...")` | 框架 | ❌ |
| 6 | 工具返回结果回灌给 Booker | 框架 | ❌ |
| 7 | **Booker 组织最终回复** | Booker | ✅ **调用 3**（视实现） |
| 8 | `event.is_final_response()` 为真 | — | ❌ |

**三个对比 §2.3 的关键数字**：

1. **模型调用 2~3 次**，而 §2.3 只有 1 次。因为每个节点都是独立的 LLM。
2. **每次转交都会换 system instruction + tool set**，ADK 会警告请求前缀变了、
   缓存失效，整个 prompt 重发。所以 §2.4 的 token 成本明显更高。
3. **你无法在代码里"拦住"第 2 步** —— 没有可插入 if 的地方。
   想加显式规则，只能改 `instruction`/`description`（软约束）或换框架。

> 本机实测输出（`gemini-3.1-flash-lite`）：
> ```
>   >>> Booking Handler Called
> FINAL: I have initiated the booking process for a hotel in Paris.
>        Do you have any specific preferences for the hotel,
>        such as dates, budget, or preferred area?
> ```
> 注意它没有直接返回工具结果，而是**在工具结果基础上又生成了一句追问** ——
> 这就是第 7 步那次额外的模型调用。

## 四个实测发现（书里没有）

### ① 书里这版代码在 ADK 2.13 上**直接跑不起来**

两个硬错：

```
RuntimeWarning: coroutine 'InMemorySessionService.create_session' was never awaited
→ An error occurred: Session not found: 614bfb9e-...
```

修法一行：`create_session` → `create_session_sync`。
（ADK 同时提供 `_sync` 变体的还有 `get_session_sync` / `list_sessions_sync` /
`delete_session_sync`，说明这不是临时方案，而是官方给同步上下文的正式接口。）

### ② 书里的 PDF 与仓库 notebook 不一致

| | PDF | 仓库 notebook |
|---|---|---|
| `run_coordinator` | `async def` + `await` | 普通 `def` |
| `main` | `async def`，`await main()` | 同步 |
| 依赖 | `nest_asyncio` | 无 |

有意思的是：**PDF 那版是对的思路但也不完整** —— 它 await 了 `create_session`，
但 `runner.run` 仍是同步生成器（不能 `await`）。所以两版都需要你动手修。

### ③ `Content` 对象没有 `.text` 属性

书里那段看着啰嗦的兼容代码：

```python
if hasattr(event.content, 'text') and event.content.text:
    final_result = event.content.text
elif event.content.parts:
    text_parts = [part.text for part in event.content.parts if part.text]
```

**不是防御性编程，是必需的。** 本机实测直接访问 `ev.content.text` 会
`AttributeError: 'Content' object has no attribute 'text'`。

### ④ 免费层限流是按**模型**分开算的

```
429 RESOURCE_EXHAUSTED ... quotaId: GenerateRequestsPerMinutePerProjectPerModel-FreeTier
quotaValue: 5, model: gemini-3.6-flash
```

三个后果：

- **5 请求/分钟**，而 §2.4 每个请求要 2~3 次调用 → **实际只能跑约 2 个用例/分钟**
- quota 是 **PerProjectPerModel**，所以**换模型 = 换配额桶**（这也是本 notebook
  默认选 `flash-lite` 的原因，lite 类通常额度更高）
- 错误里带 `retryDelay`，可以据此自动退避（代码里已实现）

## 生产化缺口

| 缺什么 | 后果 | 正解 |
|---|---|---|
| **显式兜底路径** | 模型转交失败无人处理（`unclear_handler` 是死代码） | 自己加一层 wrapper（§2.3 风格） |
| **成本控制** | 每次转交重发全量 prompt | `context_cache_config`（ADK 自己的警告也这么说） |
| **可观测性** | 不知道模型为什么选了 Booker | tracing（Ch19） |
| **路由可测性** | 没法断言"该走 Booker" | 把每次 `transfer` 事件记录下来做评测 |
| **死循环保护** | agent 之间互相转交 | `max_depth` / 步数上限 |

## 和 §2.3 该选哪个

| 场景 | 选择 |
|---|---|
| 分支少、判据明确、要可测 | **§2.3 显式路由** |
| 能力多、语义模糊、要快速扩展 | **§2.4 Auto-Flow** |
| 安全关键（机器人动作、资金） | **§2.3**，且必须能写确定性校验 |
| 面向用户的开域问答分诊 | **§2.4** |

> 最实用的答案：**两者混用**。用 §2.3 的显式分支守住安全边界，
> 在"安全落点之内"用 §2.4 的 Auto-Flow 做能力选择。

### ⑤ 无害的 stderr 噪音（别被吓到）

跑这个 notebook 时会看到这些 stderr：

```
App "InMemoryRunner" can transfer between agents but has no context_cache_config...
Deprecated. Please migrate to the async method.
Task exception was never retrieved ... AsyncClient.aclose()
```

**它们都不影响结果。** 前两条是 ADK 自己的提示（缓存配置 + 同步接口弃用警告），
第三条是 ADK 与 httpx 在收尾阶段的已知问题（异步客户端没正常关闭）。

判断依据只有一条：**看 stdout 里有没有 `>>> Booking Handler Called` 和
`Coordinator Final Response:`**。有，就是成功。

> 本 notebook 保存的输出里已清掉这些 stderr，只保留正常结果。
